# 02 · The Good

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/splevine/clustering-good-bad-beautiful/blob/main/notebooks/02_the_good.ipynb)

**A modern clustering pipeline:** UMAP → HDBSCAN → BERTopic, run on 5,000 movie overviews.

Noise becomes a first-class citizen; cluster counts come from the data, not a guess; and every cluster gets an interpretable label.

## Setup

In [ ]:
# Colab: uncomment to install
# !pip install -q sentence-transformers umap-learn hdbscan bertopic pandas pyarrow matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load the movies

In [ ]:
movies = pd.read_parquet("../data/movies.parquet")
movies = movies[movies["overview"].str.len() > 30].reset_index(drop=True)
overviews = movies["overview"].tolist()
print(f"{len(overviews):,} overviews")

## 1. Embed with sentence-transformers

`all-MiniLM-L6-v2` is a solid default — 384-D, fast, good enough for clustering short text like plot overviews.

In [ ]:
# TODO: SentenceTransformer('all-MiniLM-L6-v2').encode(overviews, show_progress_bar=True)

## 2. UMAP for structure-preserving reduction

Key parameters to discuss in the talk: `n_neighbors` (local vs. global), `min_dist` (tightness), `metric` (cosine for embeddings).

In [ ]:
# TODO: UMAP(n_neighbors=15, min_dist=0.0, metric='cosine', n_components=5) for clustering space
# TODO: UMAP(..., n_components=2) for viz

## 3. HDBSCAN for density-based clusters with noise

No `k`. Small clusters welcome. Noise labeled `-1` so you can choose what to do with it instead of the algorithm forcing a decision.

In [ ]:
# TODO: HDBSCAN(min_cluster_size=15, min_samples=5) on the 5-D UMAP space

## 4. BERTopic for interpretable labels

c-TF-IDF gives you "what words distinguish cluster X from everything else," which surfaces movie themes (heist, coming-of-age, space opera, etc.).

In [ ]:
# TODO: BERTopic(umap_model=..., hdbscan_model=..., embedding_model=...).fit_transform(overviews)

## 5. Sanity-check clusters against known genre labels

We have TMDB genre tags on each movie. They're noisy (many movies have 3+ genres) but they give a crude ground-truth signal. If our clusters align with genres at all, that's evidence of real structure.

In [ ]:
# TODO: crosstab cluster labels vs. primary genre

## Pitfalls to call out

- Over-interpreting tiny clusters.
- `min_cluster_size` is a knob with real consequences — sweep it.
- Ignoring the noise label (`-1`) vs. investigating it; noise is often the most interesting group.

Next: [`03_the_beautiful.ipynb`](03_the_beautiful.ipynb).